<a href="https://colab.research.google.com/github/HammiltonNyamache/tomato-disease-classification-cnn-vgg16/blob/main/notebooks/ML_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
# @title
# Define the paths to your train and validation directories
train_dir = os.path.join('/content/drive/MyDrive/ML PROJECT/tomatodata', 'train')
validation_dir = os.path.join('/content/drive/MyDrive/ML PROJECT/tomatodata', 'valid')

# Let's verify that the folders were created correctly
try:
    print("\nContents of the training directory:", os.listdir(train_dir))
    print("Contents of the validation directory:", os.listdir(validation_dir))

except FileNotFoundError:
    print("\n❌ Error: Could not find 'train' or 'valid' folders inside the unzipped directory.")
    print("Please check that your zip file contains these two folders directly.")


❌ Error: Could not find 'train' or 'valid' folders inside the unzipped directory.
Please check that your zip file contains these two folders directly.


#Verifying Images

In [ ]:
import os
from PIL import Image

# Use the train_dir and validation_dir paths from the previous step
train_dir = '/content/drive/MyDrive/ML PROJECT/tomatodata/train'
validation_dir = '/content/drive/MyDrive/ML PROJECT/tomatodata/valid'

def verify_images(directory):
    """
    Goes through all images in a directory and checks if they are readable.
    """
    corrupted_files = []
    for dirpath, _, filenames in os.walk(directory):
        for filename in filenames:
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                filepath = os.path.join(dirpath, filename)
                try:
                    img = Image.open(filepath) # Open the image file
                    img.verify() # Verify that it is, in fact an image
                except (IOError, SyntaxError) as e:
                    print(f"❌ Corrupted file found: {filepath}")
                    corrupted_files.append(filepath)
    return corrupted_files

print("--- Verifying Training Images ---")
corrupted_train_files = verify_images(train_dir)
print(f"Found {len(corrupted_train_files)} corrupted files in the training set.")

print("\n--- Verifying Validation Images ---")
corrupted_valid_files = verify_images(validation_dir)
print(f"Found {len(corrupted_valid_files)} corrupted files in the validation set.")


--- Verifying Training Images ---
Found 0 corrupted files in the training set.

--- Verifying Validation Images ---
Found 0 corrupted files in the validation set.


In [ ]:
os.remove('/content/drive/MyDrive/ML PROJECT/tomatodata/train/Bacterial_spot/04fcd6e4-a96e-49ba-b67a-32b88337b505___GCREC_Bact.Sp 3689.JPG')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ML PROJECT/tomatodata/train/Bacterial_spot/04fcd6e4-a96e-49ba-b67a-32b88337b505___GCREC_Bact.Sp 3689.JPG'

#Checking the Class Distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def count_files(directory):
    """
    Counts the number of files in each subdirectory of a given directory.
    """
    class_counts = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            class_counts[class_name] = len(os.listdir(class_path))
    return class_counts

# Get counts for training data
train_counts = count_files(train_dir)
df_train_counts = pd.DataFrame(list(train_counts.items()), columns=['Class', 'Count'])

# Create the bar plot
plt.figure(figsize=(12, 8))
sns.barplot(x='Count', y='Class', data=df_train_counts, palette='viridis')
plt.title('Distribution of Classes in Training Set', fontsize=16)
plt.xlabel('Number of Images', fontsize=12)
plt.ylabel('Disease Class', fontsize=12)
plt.show()

In [ ]:
import random
import cv2 # OpenCV for reading images

# Get class names from the training directory
class_names = os.listdir(train_dir)

plt.figure(figsize=(15, 15))

for i, class_name in enumerate(class_names):
    # Get a list of all image files in the class folder
    class_folder = os.path.join(train_dir, class_name)
    image_files = os.listdir(class_folder)

    # Pick a random image from the list
    random_image_file = random.choice(image_files)
    image_path = os.path.join(class_folder, random_image_file)

    # Load and display the image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert from BGR to RGB

    ax = plt.subplot(4, 3, i + 1) # Adjust the grid size (4x3) as needed
    plt.imshow(img)
    plt.title(class_name.replace('___', ' '), fontsize=12)
    plt.axis("off")

plt.tight_layout()
plt.show()

 # Prepare the Data for Training

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- Define the paths from the previous step ---
train_dir = '/content/drive/MyDrive/ML PROJECT/tomatodata/train'
validation_dir = '/content/drive/MyDrive/ML PROJECT/tomatodata/valid'

# 1. Create a generator for the training data WITH data augmentation
print("Setting up the training data generator...")
train_datagen = ImageDataGenerator(
    rescale=1./255,            # Rescale pixel values from [0, 255] to [0, 1]
    rotation_range=40,         # Randomly rotate images
    width_shift_range=0.2,     # Randomly shift images horizontally
    height_shift_range=0.2,    # Randomly shift images vertically
    shear_range=0.2,           # Apply shearing transformations
    zoom_range=0.2,            # Randomly zoom in on images
    horizontal_flip=True,      # Randomly flip images horizontally
    fill_mode='nearest'        # Strategy for filling in newly created pixels
)

# 2. Create a generator for the validation data WITHOUT data augmentation
print("Setting up the validation data generator...")
validation_datagen = ImageDataGenerator(rescale=1./255) # Only rescale the pixels

# 3. Create the actual generator objects from your directories
IMAGE_SIZE = (224, 224) # We'll use 224x224 pixels, common for these models
BATCH_SIZE = 32        # Process 32 images at a time

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'  # For multi-class classification
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# 4. Get the number of classes for the final layer of our model
num_classes = len(train_generator.class_indices)

print("\n✅ Data generators created successfully!")
print(f"Found {train_generator.samples} images belonging to {num_classes} classes for training.")
print(f"Found {validation_generator.samples} images belonging to {num_classes} classes for validation.")

# Build and Train Model 1 (A Custom CNN)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# --- Use variables defined in the previous step ---
num_classes = 11
train_generator, validation_generator

# 1. Define the model architecture
model_1 = Sequential([
    # Input layer and first convolutional block
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),

    # Second convolutional block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Third convolutional block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Flatten the results to feed into a dense layer
    Flatten(),

    # Dense (fully-connected) layers for classification
    Dense(512, activation='relu'),
    Dropout(0.5), # Dropout helps prevent overfitting
    Dense(num_classes, activation='softmax') # Output layer
])

# 2. Compile the model
# We include Precision and Recall along with Accuracy.
model_1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Display the model's architecture
model_1.summary()

# 3. Train the model
print("\n--- Starting Model 1 Training ---")
history_1 = model_1.fit(
    train_generator,
    epochs=15,  # You can start with 15 and increase if needed
    validation_data=validation_generator
)
print("\n✅ Model 1 training complete!")

In [ ]:
# Define a path for your first model
model_1_save_path = '/content/drive/MyDrive/ML PROJECT/tomato_custom_cnn_model.keras'

# Save the model
model_1.save(model_1_save_path)

print(f"✅ Model 1 successfully saved to: {model_1_save_path}")

# Evaluate Model 1 Performance

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_1.history['accuracy'])
plt.plot(history_1.history['val_accuracy'])
plt.title('Model 1 Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history_1.history['loss'])
plt.plot(history_1.history['val_loss'])
plt.title('Model 1 Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

# Build and Train Model 2 (Transfer Learning with VGG16)

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

# 1. Load the base VGG16 model without its final classification layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze the layers in the base model so we don't re-train them
base_model.trainable = False

# 3. Add our new custom classifier on top of the base model
x = base_model.output
x = Flatten()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

# 4. Create the final model
model_2 = Model(inputs=base_model.input, outputs=predictions)

# 5. Compile the model
model_2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Display the new architecture
model_2.summary()

# 6. Train the model
print("\n--- Starting Model 2 (VGG16) Training ---")
history_2 = model_2.fit(
    train_generator,
    epochs=5,
    validation_data=validation_generator
)
print("\n✅ Model 2 training complete!")

In [ ]:
# Define a path to save the model in your Google Drive
model_save_path = '/content/drive/MyDrive/ML PROJECT/tomato_vgg16_model.keras'

# Save the model
model_2.save(model_save_path)

print(f"✅ Model successfully saved to: {model_save_path}")

# Evaluate Model 2 Performance

In [ ]:
import matplotlib.pyplot as plt

# Get the training history for Model 2
history = history_2

# Plot training & validation accuracy values
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model 2 (VGG16) Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.grid(linestyle='--', linewidth=0.5)


# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model 2 (VGG16) Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.grid(linestyle='--', linewidth=0.5)


plt.tight_layout()
plt.show()

#Final Metrics Overview

In [ ]:
# --- Model 1: Custom CNN ---
best_val_accuracy_1 = max(history_1.history['val_accuracy'])
best_val_precision_1 = max(history_1.history['val_precision'])
best_val_recall_1 = max(history_1.history['val_recall'])

# --- Model 2: VGG16 ---
best_val_accuracy_2 = max(history_2.history['val_accuracy'])
best_val_precision_2 = max(history_2.history['val_precision'])
best_val_recall_2 = max(history_2.history['val_recall'])

print("--- Model 1 (Custom CNN) Best Validation Metrics ---")
print(f"Accuracy: {best_val_accuracy_1:.4f}")
print(f"Precision: {best_val_precision_1:.4f}")
print(f"Recall: {best_val_recall_1:.4f}")
print("\n" + "="*50 + "\n")
print("--- Model 2 (VGG16) Best Validation Metrics ---")
print(f"Accuracy: {best_val_accuracy_2:.4f}")
print(f"Precision: {best_val_precision_2:.4f}")
print(f"Recall: {best_val_recall_2:.4f}")

Detailed Classification Report (with F1-Score)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

y_true = []
steps = len(validation_generator)
for i in range(steps):
    _, labels = validation_generator.next()
    y_true.extend(np.argmax(labels, axis=1))

# Get class names from the generator
class_names = list(validation_generator.class_indices.keys())

# --- Model 1: Custom CNN Predictions ---
print("--- Classification Report for Model 1 (Custom CNN) ---")
y_pred_1 = np.argmax(model_1.predict(validation_generator), axis=1)
print(classification_report(y_true, y_pred_1, target_names=class_names))

# --- Model 2: VGG16 Predictions ---
print("\n" + "="*50 + "\n")
print("--- Classification Report for Model 2 (VGG16) ---")
y_pred_2 = np.argmax(model_2.predict(validation_generator), axis=1)
print(classification_report(y_true, y_pred_2, target_names=class_names))

Visual Comparison of Learning Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

# --- Accuracy Comparison ---
plt.subplot(1, 2, 1)
plt.plot(history_1.history['val_accuracy'], label='Model 1 (Custom CNN) Val Accuracy', linestyle='--')
plt.plot(history_2.history['val_accuracy'], label='Model 2 (VGG16) Val Accuracy', color='green')
plt.plot(history_1.history['accuracy'], label='Model 1 Train Accuracy', color='skyblue', alpha=0.6)
plt.plot(history_2.history['accuracy'], label='Model 2 Train Accuracy', color='lightgreen', alpha=0.8)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(linestyle='--', linewidth=0.5)

# --- Loss Comparison ---
plt.subplot(1, 2, 2)
plt.plot(history_1.history['val_loss'], label='Model 1 (Custom CNN) Val Loss', linestyle='--')
plt.plot(history_2.history['val_loss'], label='Model 2 (VGG16) Val Loss', color='red')
plt.plot(history_1.history['loss'], label='Model 1 Train Loss', color='lightcoral', alpha=0.6)
plt.plot(history_2.history['loss'], label='Model 2 Train Loss', color='orange', alpha=0.8)
plt.title('Model Loss Comparison')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()